# EduHarness — Colab API 一键跑完（混合路线）

本 notebook **只跑需要 OpenAI API 的步骤**。Remotion / Manim 渲染与 Stage 3 打包在本地完成。

## 前置（本地做一次）

```bash
cd ~/Desktop/eduharness
bash scripts/pack_colab_src.sh
# 把 eduharness-colab.zip 上传到 Google Drive:
#   MyDrive/eduharness/eduharness-colab.zip
```

Colab 左侧 🔑 **Secrets** 添加名为 **`openai`** 的 key，并勾选 Notebook access。

## 流程

| 顺序 | 在哪 | 做什么 |
|------|------|--------|
| ① | **本 notebook** `PHASE="bootstrap"` | Setup + Stage1 + 2-prepare |
| ② | **本地** | `2-render` |
| ③ | Drive 同步回来后，本 notebook `PHASE="review"` | 2-review；失败则自动 2-prepare |
| ④ | 若 review 失败 | 再本地 `2-render`，再跑 ③ |
| ⑤ | **本地** | `--stage 3` + 预览 |

默认案例：`examples/sample_request.json` → `lever_physics_01`。

## 0) 配置（每次先改这里）

In [ ]:
# === 用户配置 ===
PHASE = "review"  # "bootstrap" | "review"
# bootstrap = Setup + Stage1 + 2-prepare（第一次 / 想重新规划时）
# review    = 本地 2-render 并同步回 Drive 之后：VLM review + 失败场景 2-prepare

REQUEST_REL = "examples/sample_request.json"  # 或 examples/binary_request.json
FRESH_RUN = False  # True = 删除该 request 的旧 Drive run 目录后重来
MAX_REVIEW_ROUNDS = 2  # review 阶段最多自动 prepare 修复轮次提示用

assert PHASE in {"bootstrap", "review"}, PHASE

## 1) Setup（解压源码 + API Key + 依赖）

每个 Colab 内核独立，**每次打开本 notebook 都要跑本格**。

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)

DRIVE_ZIP = Path("/content/drive/MyDrive/eduharness/eduharness-colab.zip")
REPO = Path("/content")
RUN_ROOT = Path("/content/drive/MyDrive/eduharness/runs")

if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(
        f"请先把 zip 上传到 Drive:\n  {DRIVE_ZIP}\n"
        "本地执行: bash scripts/pack_colab_src.sh"
    )

print("Unzipping source…")
subprocess.check_call(["unzip", "-o", str(DRIVE_ZIP), "-d", "/content"])

if not (REPO / "eduharness" / "pipeline.py").is_file():
    raise FileNotFoundError(f"解压后未找到 {REPO}/eduharness/pipeline.py")

RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ["EDUHARNESS_TEXT_MODEL"] = "gpt-5"
os.environ["EDUHARNESS_VISION_MODEL"] = "gpt-5"
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")

from eduharness.colab_session import ensure_api_key

ensure_api_key()

print("Installing Python deps…")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")]
)

# Review 需要抽帧 / interactive 截图
if PHASE == "review":
    print("Installing review deps (ffmpeg + playwright)…")
    subprocess.check_call(["apt-get", "-qq", "update"])
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright"])
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])

REQUEST_PATH = (REPO / REQUEST_REL).resolve()
if not REQUEST_PATH.is_file():
    raise FileNotFoundError(REQUEST_PATH)

cfg_blob = {
    "REPO": str(REPO),
    "RUN_DIR": str(RUN_ROOT),
    "REQUEST": str(REQUEST_PATH),
    "PHASE": PHASE,
}
Path("/content/eduharness_colab_env.json").write_text(
    json.dumps(cfg_blob, indent=2), encoding="utf-8"
)
print(json.dumps(cfg_blob, indent=2, ensure_ascii=False))
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## 2) API：Stage 1 + Stage 2-prepare（仅 `PHASE="bootstrap"`）

产出写在 Drive：`MyDrive/eduharness/runs/<request_id>/`

In [ ]:
from eduharness.config import Config
from eduharness.pipeline import load_request, stage1, stage2_prepare

if PHASE != "bootstrap":
    print(f"Skip bootstrap (PHASE={PHASE!r}). Jump to section 4 review cell.")
else:
    request = load_request(REQUEST_PATH)
    notes = REPO / "examples" / "binary_notes.txt"
    if "binary" in REQUEST_PATH.name and notes.is_file() and not request.source_text:
        request.source_text = notes.read_text(encoding="utf-8")

    run_dir = RUN_ROOT / request.request_id
    if FRESH_RUN and run_dir.exists():
        print("Removing old run:", run_dir)
        shutil.rmtree(run_dir)

    cfg = Config.from_env(request, run_dir=run_dir)
    cfg.max_review_rounds = max(1, int(MAX_REVIEW_ROUNDS))

    print("=== Stage 1: plan ===")
    blueprint = stage1(cfg)
    for s in blueprint.scenes:
        print(f"  {s.id:22} [{s.scene_type.value:12}] {s.title}")

    print("=== Stage 2-prepare (API, includes SceneRepair-aware retries later) ===")
    prepared = stage2_prepare(cfg)
    print(f"Prepared {len(prepared)} scenes → {cfg.prepared_path}")
    print(f"Run dir (Drive): {cfg.run_dir}")

## 3) ⏸️ 停一下：本地 `2-render`（无 API）

把 Drive 里的 run 同步到本机（Finder / Drive 桌面客户端 / 手动下载 zip），然后：

```bash
cd ~/Desktop/eduharness
source .venv/bin/activate

# 若 run 在别处，拷到:
#   runs/lever_physics_01/

python -m eduharness \
  --request-json examples/sample_request.json \
  --run-dir runs/lever_physics_01 \
  --stage 2-render
```

渲染完成后，把本地 `runs/<id>/`（至少含 `media/`、`interactive/`、`rendered_scenes.json`、`work/`）**同步回**
`MyDrive/eduharness/runs/<id>/`。

然后回到本 notebook：把上面配置里的 `PHASE = "review"`，**Runtime → Restart** 后从 Setup 格重新跑到 Review 格。

In [ ]:
# 打印给本机用的命令（bootstrap 后看这里即可）
from eduharness.pipeline import load_request

request = load_request(REQUEST_PATH)
rid = request.request_id
print(f"""
======== LOCAL 2-RENDER ========
cd ~/Desktop/eduharness && source .venv/bin/activate

# Sync Drive → local first, then:
python -m eduharness \\
  --request-json {REQUEST_REL} \\
  --run-dir runs/{rid} \\
  --stage 2-render

# Sync local runs/{rid} back to Drive, then set PHASE='review' in this notebook.
================================
""")

## 4) API：Stage 2-review + 失败场景自动 2-prepare（仅 `PHASE="review"`）

会写出结构化 `SceneRepair`（diff ops）到 `debug/*_repair_r*.json`。  
若有失败场景，会立刻再跑一轮 `2-prepare`（patch-first），然后提示你再本地 `2-render`。

In [ ]:
from eduharness.config import Config
from eduharness.pipeline import load_request, stage2_prepare, stage2_review

if PHASE != "review":
    print(f"Skip review (PHASE={PHASE!r}). After local 2-render, set PHASE='review' and rerun.")
else:
    request = load_request(REQUEST_PATH)
    cfg = Config.from_env(request, run_dir=RUN_ROOT / request.request_id)
    cfg.max_review_rounds = max(1, int(MAX_REVIEW_ROUNDS))

    assert cfg.blueprint_path.is_file(), "Missing blueprint — run PHASE='bootstrap' first"
    assert cfg.rendered_path.is_file(), (
        "Missing rendered_scenes.json — sync local 2-render output back to Drive first"
    )

    print("=== Stage 2-review (VLM → SceneRepair) ===")
    rendered = stage2_review(cfg)
    failed = []
    for r in rendered:
        repair = r.repair
        action = repair.fix_action if repair else "-"
        n_ops = len(repair.ops) if repair else 0
        print(
            f"  {r.scene_id:22} pass={r.review_passed!s:5} "
            f"action={action:16} ops={n_ops} src={r.src}"
        )
        if not r.review_passed and (repair is None or repair.needs_repair()):
            failed.append(r.scene_id)

    if not failed:
        print("\nAll scenes passed review.")
        print("Next: sync Drive → local, then run Stage 3 locally (see next cell).")
    else:
        print("\nFailed scenes:", failed)
        print("=== Auto 2-prepare for failed scenes (diff patch / targeted regen) ===")
        prepared = stage2_prepare(cfg)
        for p in prepared:
            if p.scene_id in failed:
                mode = (p.debug or {}).get("repair_mode", "?")
                applied = (p.debug or {}).get("ops_applied", [])
                print(f"  {p.scene_id:22} repair_mode={mode} applied={applied}")
        print(
            f"""
======== LOCAL 2-RENDER (again) ========
cd ~/Desktop/eduharness && source .venv/bin/activate
python -m eduharness \\
  --request-json {REQUEST_REL} \\
  --run-dir runs/{request.request_id} \\
  --stage 2-render

# Sync back to Drive, then re-run this review cell (PHASE='review').
========================================
"""
        )

## 5) 本地 Stage 3 + 预览（无 API）

全部 `review_passed=true` 且已同步回本地后：

```bash
cd ~/Desktop/eduharness
source .venv/bin/activate

python -m eduharness \
  --request-json examples/sample_request.json \
  --run-dir runs/lever_physics_01 \
  --stage 3

cd runs/lever_physics_01/bundle && python3 -m http.server 8765
# 浏览器打开 http://127.0.0.1:8765
```

In [ ]:
from eduharness.pipeline import load_request

request = load_request(REQUEST_PATH)
rid = request.request_id
print(f"""
======== LOCAL STAGE 3 + PREVIEW ========
cd ~/Desktop/eduharness && source .venv/bin/activate

# Sync Drive runs/{rid} → local runs/{rid} first
python -m eduharness \\
  --request-json {REQUEST_REL} \\
  --run-dir runs/{rid} \\
  --stage 3

cd runs/{rid}/bundle && python3 -m http.server 8765
# open http://127.0.0.1:8765
=========================================
""")

# 若已在 Drive 上有 rendered，快速看过审状态
run_dir = RUN_ROOT / rid
rendered_path = run_dir / "rendered_scenes.json"
if rendered_path.is_file():
    data = json.loads(rendered_path.read_text(encoding="utf-8"))
    ok = all(item.get("review_passed") for item in data)
    print(f"Drive review all_pass={ok}  scenes={len(data)}  path={rendered_path}")
    for item in data:
        print(f"  {item['scene_id']:22} pass={item.get('review_passed')}")
else:
    print("No rendered_scenes.json on Drive yet.")